In [1]:

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.utils import to_undirected
from torch_geometric.nn import GCN2Conv

d:\elliptic\venv1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

if hasattr(torch, "xpu") and torch.xpu.is_available():
    device = torch.device("xpu")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("\nDevice:", device)


Device: xpu


In [3]:

DATA_DIR = r"D:\elliptic\Elliptic_Dataset"
TXS_FEATURES_FILE = "txs_features.csv"
TXS_CLASSES_FILE  = "txs_classes.csv"
TXS_EDGELIST_FILE = "txs_edgelist.csv"

df_feat = pd.read_csv(f"{DATA_DIR}/{TXS_FEATURES_FILE}")
df_cls  = pd.read_csv(f"{DATA_DIR}/{TXS_CLASSES_FILE}")
df_edge = pd.read_csv(f"{DATA_DIR}/{TXS_EDGELIST_FILE}")

print("txs_features shape:", df_feat.shape)
print("txs_classes  shape:", df_cls.shape)
print("txs_edgelist shape:", df_edge.shape)


df = df_feat.merge(df_cls, on="txId", how="left")

df = df.dropna()
df["class"] = df["class"].astype(int)

print("\nClass raw distribution (1=licit, 2=illicit, 3=unknown):")
print(df["class"].value_counts())

df = df[df["class"] != 3].copy()

label_map = {1: 0, 2: 1}
df["label"] = df["class"].map(label_map)

print("\nLabel distribution (0=licit,1=illicit):")
print(df["label"].value_counts())



txs_features shape: (203769, 184)
txs_classes  shape: (203769, 2)
txs_edgelist shape: (234355, 2)

Class raw distribution (1=licit, 2=illicit, 3=unknown):
class
3    156759
2     41500
1      4545
Name: count, dtype: int64

Label distribution (0=licit,1=illicit):
label
1    41500
0     4545
Name: count, dtype: int64


In [4]:

cols_to_exclude = {"txId", "class", "label", "Time step"}


numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in numeric_cols if c not in cols_to_exclude]

print("\nNumber of feature:", len(feature_cols))


X_all = df[feature_cols].astype(float).values
y_all = df["label"].values
txid_all = df["txId"].astype(int).values
time_all = df["Time step"].values



Number of feature: 182


In [5]:


unique_ts = np.sort(df["Time step"].unique())
print("\nTime steps unique:", unique_ts)

train_ts = unique_ts[unique_ts <= 30]
val_ts   = unique_ts[(unique_ts > 30) & (unique_ts <=34)]
test_ts  = unique_ts[unique_ts > 34]


print("\nTime-step TRAIN:", train_ts[0], "->", train_ts[-1])
print("Time-step VAL  :", val_ts[0],   "->", val_ts[-1])
print("Time-step TEST :", test_ts[0],  "->", test_ts[-1])

train_mask = np.isin(time_all, train_ts)
val_mask   = np.isin(time_all, val_ts)
test_mask  = np.isin(time_all, test_ts)

# raw split
X_train_raw = X_all[train_mask]
X_val_raw   = X_all[val_mask]
X_test_raw  = X_all[test_mask]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val   = scaler.transform(X_val_raw)
X_test  = scaler.transform(X_test_raw)
# scaler_train = StandardScaler()


y_train = y_all[train_mask]
y_val   = y_all[val_mask]
y_test  = y_all[test_mask]
txid_train = txid_all[train_mask]
txid_val   = txid_all[val_mask]
txid_test  = txid_all[test_mask]

print("\nSIZE:")
print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)
print("\nLabel distribution in the training set:")
print(pd.Series(y_train).value_counts())
print("\nLabel distribution in the validation set:")
print(pd.Series(y_val).value_counts())
print("\nLabel distribution in the testing set:")
print(pd.Series(y_test).value_counts())


Time steps unique: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48
 49]

Time-step TRAIN: 1 -> 30
Time-step VAL  : 31 -> 34
Time-step TEST : 35 -> 49

SIZE:
Train: (26750, 182) Val: (2949, 182) Test: (16346, 182)

Label distribution in the training set:
1    23796
0     2954
Name: count, dtype: int64

Label distribution in the validation set:
1    2441
0     508
Name: count, dtype: int64

Label distribution in the testing set:
1    15263
0     1083
Name: count, dtype: int64


In [6]:


def build_edge_index(df_edges, valid_txids):

    node_ids = np.asarray(valid_txids, dtype=np.int64)
    id2idx = {tid: i for i, tid in enumerate(node_ids)}

    mask = df_edges["txId1"].isin(node_ids) & df_edges["txId2"].isin(node_ids)
    edges_sub = df_edges.loc[mask, ["txId1", "txId2"]]

    if len(edges_sub) == 0:

        edges_idx = np.zeros((2, 0), dtype=np.int64)
        edge_index = torch.tensor(edges_idx, dtype=torch.long)
        return edge_index

    src_idx = edges_sub["txId1"].map(id2idx).values
    dst_idx = edges_sub["txId2"].map(id2idx).values
    edges_idx = np.vstack([src_idx, dst_idx])

    edge_index = torch.tensor(edges_idx, dtype=torch.long)
    edge_index = to_undirected(edge_index)
    return edge_index

edge_index_train = build_edge_index(df_edge, txid_train)
edge_index_val   = build_edge_index(df_edge, txid_val)
edge_index_test  = build_edge_index(df_edge, txid_test)

print("\nTrain edges:", edge_index_train.size(1),
      "Val edges:", edge_index_val.size(1),
      "Test edges:", edge_index_test.size(1))


Train edges: 40860 Val edges: 4732 Test edges: 27232


In [7]:


X_train_gcn = torch.tensor(X_train, dtype=torch.float)
X_val_gcn   = torch.tensor(X_val,   dtype=torch.float)
X_test_gcn  = torch.tensor(X_test,  dtype=torch.float)

y_train_gcn = torch.tensor(y_train, dtype=torch.long)
y_val_gcn   = torch.tensor(y_val,   dtype=torch.long)
y_test_gcn  = torch.tensor(y_test,  dtype=torch.long)

train_data = Data(x=X_train_gcn, edge_index=edge_index_train, y=y_train_gcn)
val_data   = Data(x=X_val_gcn,   edge_index=edge_index_val,   y=y_val_gcn)
test_data  = Data(x=X_test_gcn,  edge_index=edge_index_test,  y=y_test_gcn)


train_data.node_ids = torch.tensor(txid_train, dtype=torch.long)
val_data.node_ids   = torch.tensor(txid_val,   dtype=torch.long)
test_data.node_ids  = torch.tensor(txid_test,  dtype=torch.long)

print("\ntrain_data:", train_data)
print("val_data  :", val_data)
print("test_data :", test_data)

print("\nCheck NaN in train features:", torch.isnan(train_data.x).any().item())
print("Check Inf in train features:", torch.isinf(train_data.x).any().item())


train_data: Data(x=[26750, 182], edge_index=[2, 40860], y=[26750], node_ids=[26750])
val_data  : Data(x=[2949, 182], edge_index=[2, 4732], y=[2949], node_ids=[2949])
test_data : Data(x=[16346, 182], edge_index=[2, 27232], y=[16346], node_ids=[16346])

Check NaN in train features: False
Check Inf in train features: False


In [8]:
def build_trainval_graph(train_data, val_data, df_edge, device=None):

    tx_train = train_data.node_ids.detach().cpu().numpy()
    tx_val   = val_data.node_ids.detach().cpu().numpy()
    tx_tv    = np.concatenate([tx_train, tx_val]) 


    x_tv = torch.cat([train_data.x, val_data.x], dim=0)
    y_tv = torch.cat([train_data.y, val_data.y], dim=0)


    tx2new = {int(t): i for i, t in enumerate(tx_tv)}


    c0, c1 = df_edge.columns[:2]  
    sub = df_edge[df_edge[c0].isin(tx2new) & df_edge[c1].isin(tx2new)].copy()


    src = sub[c0].map(tx2new).to_numpy(dtype=np.int64)
    dst = sub[c1].map(tx2new).to_numpy(dtype=np.int64)

    edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
    edge_index = to_undirected(edge_index)  


    tv_data = Data(x=x_tv, y=y_tv, edge_index=edge_index)
    tv_data.node_ids = torch.tensor(tx_tv, dtype=torch.long)

    if device is not None:
        tv_data = tv_data.to(device)

    return tv_data



trainval_data = build_trainval_graph(train_data, val_data, df_edge, device=device)

print("trainval_data:", trainval_data)
print("N_trainval nodes:", trainval_data.num_nodes)
print("N_trainval edges:", trainval_data.edge_index.size(1))

trainval_data: Data(x=[29699, 182], edge_index=[2, 45592], y=[29699], node_ids=[29699])
N_trainval nodes: 29699
N_trainval edges: 45592


In [9]:



class_sample_count = torch.bincount(train_data.y, minlength=2).float()
eps = 1e-8
inv_freq = 1.0 / (class_sample_count + eps)
norm_inv_freq = inv_freq / inv_freq.min()

print("Class counts (train):", class_sample_count.tolist())
print("Class weights (inv_freq normalized):", norm_inv_freq.tolist())

Class counts (train): [2954.0, 23796.0]
Class weights (inv_freq normalized): [8.05551815032959, 1.0]


In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.utils import scatter  


def create_ffn(in_dim: int, hidden_units: int, dropout_rate: float) -> nn.Module:

    return nn.Sequential(
        nn.Linear(in_dim, hidden_units),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout_rate),
        nn.Linear(hidden_units, hidden_units),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout_rate),
    )


class GraphConvLayer(nn.Module):


    def __init__(
        self,
        hidden_units: int,
        dropout_rate: float = 0.2,
        aggregation_type: str = "mean",    
        combination_type: str = "concat",
        normalize: bool = False,
    ):
        super().__init__()
        self.hidden_units = hidden_units
        self.dropout_rate = dropout_rate
        self.aggregation_type = aggregation_type
        self.combination_type = combination_type
        self.normalize = normalize

        self.ffn_prepare = create_ffn(hidden_units, hidden_units, dropout_rate)

        if self.combination_type == "gru":
            self.update_gru = nn.GRU(
                input_size=hidden_units,
                hidden_size=hidden_units,
                batch_first=True,
                dropout=dropout_rate if dropout_rate > 0 else 0.0,
            )
            self.update_mlp = None
        else:
            in_dim = 2 * hidden_units if self.combination_type == "concat" else hidden_units
            self.update_mlp = create_ffn(in_dim, hidden_units, dropout_rate)
            self.update_gru = None

    def forward(self, node_repr: torch.Tensor, edge_index: torch.Tensor, edge_weight: torch.Tensor | None):

        src, dst = edge_index[0], edge_index[1]
        neigh_repr = node_repr[src]  


        messages = self.ffn_prepare(neigh_repr) 
        if edge_weight is not None:
            messages = messages * edge_weight.view(-1, 1)


        reduce = {"sum": "sum", "mean": "mean", "max": "max"}[self.aggregation_type]
        aggregated = scatter(messages, dst, dim=0, dim_size=node_repr.size(0), reduce=reduce) 


        if self.combination_type == "gru":
            seq = torch.stack([node_repr, aggregated], dim=1)   
            out, _ = self.update_gru(seq)
            node_emb = out[:, -1, :]                             
        elif self.combination_type == "concat":
            h = torch.cat([node_repr, aggregated], dim=1)     
            node_emb = self.update_mlp(h)                      
        elif self.combination_type == "add":
            h = node_repr + aggregated                        
            node_emb = self.update_mlp(h)                   
        else:
            raise ValueError(f"Invalid combination_type: {self.combination_type}")

        if self.normalize:
            node_emb = F.normalize(node_emb, p=2, dim=-1)
        return node_emb


class GNNNodeClassifier(nn.Module):


    def __init__(
        self,
        in_features: int,
        num_classes: int,
        hidden_units: int,
        aggregation_type: str = "sum",
        combination_type: str = "concat",
        dropout_rate: float = 0.2,
        normalize: bool = True,
    ):
        super().__init__()


        self.preprocess = create_ffn(in_features, hidden_units, dropout_rate)

        self.conv1 = GraphConvLayer(
            hidden_units=hidden_units,
            dropout_rate=dropout_rate,
            aggregation_type=aggregation_type,
            combination_type=combination_type,
            normalize=normalize,
        )
        self.conv2 = GraphConvLayer(
            hidden_units=hidden_units,
            dropout_rate=dropout_rate,
            aggregation_type=aggregation_type,
            combination_type=combination_type,
            normalize=normalize,
        )

        self.postprocess = create_ffn(hidden_units, hidden_units, dropout_rate)
        self.compute_logits = nn.Linear(hidden_units, num_classes)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, edge_weight: torch.Tensor | None = None):

        if edge_weight is None:
            edge_weight = torch.ones(edge_index.size(1), device=x.device, dtype=x.dtype)
        edge_weight = edge_weight / (edge_weight.sum() + 1e-12)

        h = self.preprocess(x)                    
        h1 = self.conv1(h, edge_index, edge_weight) 
        h = h1 + h                                  
        h2 = self.conv2(h, edge_index, edge_weight)  
        h = h2 + h                                 

        h = self.postprocess(h)                    
        logits = self.compute_logits(h)          
        return logits


In [11]:



class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.reduction = reduction

    def forward(self, logits, target):
        logp = F.log_softmax(logits, dim=1)
        p = logp.exp()

        target = target.view(-1, 1)
        logp_t = logp.gather(1, target).squeeze(1)
        p_t    = p.gather(1, target).squeeze(1)

        focal_term = (1 - p_t) ** self.gamma
        loss = - focal_term * logp_t

        if self.weight is not None:
            w = self.weight[target.squeeze(1)].view(-1)
            loss = loss * w

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        else:
            return loss


def train_one_config(config, loss_type="ce", class_weights=None,
                     max_epochs=1000, patience=30, verbose=False):
    in_dim  = train_data.x.size(1)
    out_dim = 2

    model = GNNNodeClassifier(
        in_features = train_data.x.size(1),
        num_classes = 2,
        hidden_units = config["hid_dim"],    
        dropout_rate = config["dropout"],    
        aggregation_type = "mean",
        combination_type = "concat",
        normalize = True
    ).to(device)

    cw = class_weights.to(device) if class_weights is not None else None

    if loss_type == "ce":
        crit = nn.CrossEntropyLoss(weight=cw)
    elif loss_type == "focal":
        crit = FocalLoss(gamma=config.get("gamma", 2.0), weight=cw)
    else:
        raise ValueError("loss_type must be 'ce' or 'focal'")

    opt = torch.optim.Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    def eval_for_search(data):
        model.eval()
        with torch.no_grad():
            out = model(data.x.to(device), data.edge_index.to(device))
            loss = crit(out, data.y.to(device)).item()
            preds = out.argmax(dim=1).cpu().numpy()
            y_true = data.y.cpu().numpy()
            macro_f1 = f1_score(y_true, preds, average="macro", zero_division=0)
        return loss, macro_f1

    best_state = None
    best_val_macro = -1.0
    patience_counter = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        opt.zero_grad()
        out = model(train_data.x.to(device), train_data.edge_index.to(device))
        loss_train = crit(out, train_data.y.to(device))
        loss_train.backward()
        opt.step()

        val_loss, val_macro = eval_for_search(val_data)

        if verbose and (epoch % 20 == 0 or epoch == 1):
            print(f"[{config.get('name','?')}] Epoch {epoch:03d} "
                  f"- train_loss={loss_train.item():.4f} "
                  f"- val_loss={val_loss:.4f} "
                  f"- val_macro={val_macro:.4f}")

        if val_macro > best_val_macro + 1e-4:
            best_val_macro = val_macro
            best_state = torch.save(model.state_dict(), "tmp_best_tx_model.pt")
            best_state = torch.load("tmp_best_tx_model.pt", map_location="cpu")
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            if verbose:
                print(f"Early stop (no improve {patience} epochs)")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_val_macro

In [12]:


@torch.no_grad()
def tune_threshold_on_val(model, data, thresholds=None):
    model.eval()
    if thresholds is None:
        thresholds = np.linspace(0.1, 0.9, 17)

    out = model(data.x.to(device), data.edge_index.to(device))
    probs = F.softmax(out, dim=1)[:, 1].cpu().numpy() 
    y_true = data.y.cpu().numpy()

    results = []
    for t in thresholds:
        preds = (probs >= t).astype(int)
        macro_f1 = f1_score(y_true, preds, average="macro", zero_division=0)
        results.append((t, macro_f1))

    best_t, best_macro = max(results, key=lambda x: x[1])
    print("\nThreshold search (val):")
    for t, m in results:
        print(f"  t={t:.2f}  macro-F1={m:.6f}")
    print(f"\nBest threshold on VAL: t={best_t:.2f}, macro-F1={best_macro:.6f}")
    return best_t, best_macro


@torch.no_grad()
def evaluate_with_threshold(model, data, threshold=0.5, name="SET"):
    model.eval()
    out = model(data.x.to(device), data.edge_index.to(device))
    probs = F.softmax(out, dim=1)[:, 1].cpu().numpy()
    y_true = data.y.cpu().numpy()
    

    try:
        auc = roc_auc_score(y_true, probs) if len(np.unique(y_true)) > 1 else np.nan
    except Exception:
        auc = np.nan

    preds = (probs >= threshold).astype(int)

    f1_scam  = f1_score(y_true, preds, pos_label=1, zero_division=0)
    micro_f1 = f1_score(y_true, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(y_true, preds, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, preds, labels=[0, 1])

    print(f"\n{name} with threshold={threshold:.2f}")
    print(f"ROC-AUC: {auc:.6f}")
    print(f"F1 (illicit=1): {f1_scam:.6f}")
    print(f"Micro-F1: {micro_f1:.6f}")
    print(f"Macro-F1: {macro_f1:.6f}")
    print("Confusion matrix:\n", cm)
    print("\nClassification report:")
    print(classification_report(y_true, preds, digits=6))

In [13]:


arch_space = [
    {"name": "arch1", "hid_dim": 64,  "num_layers": 8,  "dropout": 0.5,
     "lr": 1e-2, "weight_decay": 5e-4, "alpha": 0.1},
    {"name": "arch2", "hid_dim": 64,  "num_layers": 16, "dropout": 0.5,
     "lr": 1e-2, "weight_decay": 5e-4, "alpha": 0.1},
    {"name": "arch3", "hid_dim": 128, "num_layers": 16, "dropout": 0.5,
     "lr": 5e-3, "weight_decay": 5e-4, "alpha": 0.1},
    {"name": "arch4", "hid_dim": 128, "num_layers": 32, "dropout": 0.5,
     "lr": 5e-3, "weight_decay": 1e-3,"alpha": 0.1},
]

loss_configs = [
    {"name": "CE_no_weight",   "loss_type": "ce",
     "class_weights": torch.tensor([1.0, 1.0])},
    {"name": "CE_inv_freq",    "loss_type": "ce",
     "class_weights": norm_inv_freq},
    {"name": "Focal_gamma1.5", "loss_type": "focal",
     "class_weights": norm_inv_freq, "gamma": 1.5},
    {"name": "Focal_gamma2.0", "loss_type": "focal",
     "class_weights": norm_inv_freq, "gamma": 2.0},
]

best_model = None
best_arch  = None
best_loss_cfg = None
best_val_macro = -1.0

for arch in arch_space:
    for lc in loss_configs:
        cfg = arch.copy()
        cfg["name"] = arch["name"] + "_" + lc["name"]
        if lc["loss_type"] == "focal":
            cfg["gamma"] = lc["gamma"]

        print(f"\n=== Training config: {cfg['name']} (loss={lc['loss_type']}) ===")
        model_cfg, val_macro = train_one_config(
            cfg,
            loss_type=lc["loss_type"],
            class_weights=lc["class_weights"],
            max_epochs=1000,
            patience=40,
            verbose=False,
        )
        print(f"Config {cfg['name']}  val_macro = {val_macro:.4f}")

        if val_macro > best_val_macro:
            best_val_macro = val_macro
            best_model = model_cfg
            best_arch = arch
            best_loss_cfg = lc

print("\n>>> BEST CONFIG OVERALL")
print("Best arch:", best_arch)
print("Best loss config:", best_loss_cfg)
print("Best val macro-F1:", best_val_macro)


=== Training config: arch1_CE_no_weight (loss=ce) ===
Config arch1_CE_no_weight  val_macro = 0.8947

=== Training config: arch1_CE_inv_freq (loss=ce) ===
Config arch1_CE_inv_freq  val_macro = 0.9175

=== Training config: arch1_Focal_gamma1.5 (loss=focal) ===
Config arch1_Focal_gamma1.5  val_macro = 0.9143

=== Training config: arch1_Focal_gamma2.0 (loss=focal) ===
Config arch1_Focal_gamma2.0  val_macro = 0.9140

=== Training config: arch2_CE_no_weight (loss=ce) ===
Config arch2_CE_no_weight  val_macro = 0.4529

=== Training config: arch2_CE_inv_freq (loss=ce) ===
Config arch2_CE_inv_freq  val_macro = 0.9056

=== Training config: arch2_Focal_gamma1.5 (loss=focal) ===
Config arch2_Focal_gamma1.5  val_macro = 0.9207

=== Training config: arch2_Focal_gamma2.0 (loss=focal) ===
Config arch2_Focal_gamma2.0  val_macro = 0.9144

=== Training config: arch3_CE_no_weight (loss=ce) ===
Config arch3_CE_no_weight  val_macro = 0.8905

=== Training config: arch3_CE_inv_freq (loss=ce) ===
Config arch3_

In [14]:


best_threshold, _ = tune_threshold_on_val(best_model, val_data)

evaluate_with_threshold(best_model, train_data, best_threshold, name="TRAIN")
evaluate_with_threshold(best_model, val_data,   best_threshold, name="VAL")
evaluate_with_threshold(best_model, test_data,  best_threshold, name="TEST")


Threshold search (val):
  t=0.10  macro-F1=0.866890
  t=0.15  macro-F1=0.886122
  t=0.20  macro-F1=0.900686
  t=0.25  macro-F1=0.901360
  t=0.30  macro-F1=0.909903
  t=0.35  macro-F1=0.915423
  t=0.40  macro-F1=0.919449
  t=0.45  macro-F1=0.926007
  t=0.50  macro-F1=0.929744
  t=0.55  macro-F1=0.928487
  t=0.60  macro-F1=0.922964
  t=0.65  macro-F1=0.916883
  t=0.70  macro-F1=0.910836
  t=0.75  macro-F1=0.897907
  t=0.80  macro-F1=0.872927
  t=0.85  macro-F1=0.820285
  t=0.90  macro-F1=0.723943

Best threshold on VAL: t=0.50, macro-F1=0.929744

TRAIN with threshold=0.50
ROC-AUC: 0.999384
F1 (illicit=1): 0.992958
Micro-F1: 0.987551
Macro-F1: 0.969663
Confusion matrix:
 [[ 2938    16]
 [  317 23479]]

Classification report:
              precision    recall  f1-score   support

           0   0.902611  0.994584  0.946368      2954
           1   0.999319  0.986678  0.992958     23796

    accuracy                       0.987551     26750
   macro avg   0.950965  0.990631  0.969663     2

In [15]:


txid_trainval = np.concatenate([txid_train, txid_val])
X_trainval    = np.vstack([X_train, X_val])
y_trainval    = np.concatenate([y_train, y_val])

print("Train+Val shapes:", X_trainval.shape, y_trainval.shape, txid_trainval.shape)

intersect = np.intersect1d(txid_train, txid_val)
print("Overlap txId(train, val) =", intersect.size)


edge_index_trainval = build_edge_index(df_edge, txid_trainval)
print("Train+Val edges:", edge_index_trainval.size(1))


X_trainval_gcn = torch.tensor(X_trainval, dtype=torch.float)
y_trainval_gcn = torch.tensor(y_trainval, dtype=torch.long)

trainval_data = Data(x=X_trainval_gcn, edge_index=edge_index_trainval, y=y_trainval_gcn)
trainval_data.node_ids = torch.tensor(txid_trainval, dtype=torch.long)

print("trainval_data:", trainval_data)
print("Check NaN in trainval features:", torch.isnan(trainval_data.x).any().item())
print("Check Inf in trainval features:", torch.isinf(trainval_data.x).any().item())


Train+Val shapes: (29699, 182) (29699,) (29699,)
Overlap txId(train, val) = 0
Train+Val edges: 45592
trainval_data: Data(x=[29699, 182], edge_index=[2, 45592], y=[29699], node_ids=[29699])
Check NaN in trainval features: False
Check Inf in trainval features: False


In [16]:

best_threshold_pre, _ = tune_threshold_on_val(best_model, val_data)
print("Saved best_threshold (from VAL pre-final-train):", best_threshold_pre)


_train_backup, _val_backup = train_data, val_data


train_data = trainval_data
val_data   = trainval_data   

final_model, _ = train_one_config(
    best_arch,
    loss_type=best_loss_cfg["loss_type"],
    class_weights=best_loss_cfg["class_weights"],
    max_epochs=1000,
    patience=40,
    verbose=True,
)

train_data, val_data = _train_backup, _val_backup





Threshold search (val):
  t=0.10  macro-F1=0.866890
  t=0.15  macro-F1=0.886122
  t=0.20  macro-F1=0.900686
  t=0.25  macro-F1=0.901360
  t=0.30  macro-F1=0.909903
  t=0.35  macro-F1=0.915423
  t=0.40  macro-F1=0.919449
  t=0.45  macro-F1=0.926007
  t=0.50  macro-F1=0.929744
  t=0.55  macro-F1=0.928487
  t=0.60  macro-F1=0.922964
  t=0.65  macro-F1=0.916883
  t=0.70  macro-F1=0.910836
  t=0.75  macro-F1=0.897907
  t=0.80  macro-F1=0.872927
  t=0.85  macro-F1=0.820285
  t=0.90  macro-F1=0.723943

Best threshold on VAL: t=0.50, macro-F1=0.929744
Saved best_threshold (from VAL pre-final-train): 0.5
[arch3] Epoch 001 - train_loss=0.3217 - val_loss=0.3160 - val_macro=0.4691
[arch3] Epoch 020 - train_loss=0.0970 - val_loss=0.0783 - val_macro=0.9134
[arch3] Epoch 040 - train_loss=0.0582 - val_loss=0.0467 - val_macro=0.9532
[arch3] Epoch 060 - train_loss=0.0473 - val_loss=0.0335 - val_macro=0.9493
[arch3] Epoch 080 - train_loss=0.0395 - val_loss=0.0277 - val_macro=0.9727
[arch3] Epoch 100 - t

In [17]:

evaluate_with_threshold(final_model, test_data, best_threshold_pre, name="TEST (final train on train+val)")


TEST (final train on train+val) with threshold=0.50
ROC-AUC: 0.908113
F1 (illicit=1): 0.979841
Micro-F1: 0.961764
Macro-F1: 0.804900
Confusion matrix:
 [[  532   551]
 [   74 15189]]

Classification report:
              precision    recall  f1-score   support

           0   0.877888  0.491228  0.629959      1083
           1   0.964994  0.995152  0.979841     15263

    accuracy                       0.961764     16346
   macro avg   0.921441  0.743190  0.804900     16346
weighted avg   0.959222  0.961764  0.956659     16346



In [18]:

import os, json
import joblib
import numpy as np
import torch
import torch.nn.functional as F

GNN_SAVE_DIR = "gnn_saved"
os.makedirs(GNN_SAVE_DIR, exist_ok=True)


ckpt = {
    "model_class": final_model.__class__.__name__,
    "state_dict": final_model.state_dict(),
    "best_arch": best_arch,
    "best_loss_cfg": best_loss_cfg,
    "feature_cols": feature_cols,
    "threshold_from_val": float(best_threshold_pre),
}
ckpt_path = os.path.join(GNN_SAVE_DIR, "txs_gnn_checkpoint.pt")
torch.save(ckpt, ckpt_path)
print("Saved checkpoint:", ckpt_path)

joblib.dump(scaler,  os.path.join(GNN_SAVE_DIR, "scaler.pkl"))
print("Saved scalers (*.pkl) into", GNN_SAVE_DIR)


final_model.eval()
with torch.no_grad():
    out = final_model(test_data.x.to(device), test_data.edge_index.to(device))
    proba_test = F.softmax(out, dim=1)[:, 1].detach().cpu().numpy()

txid_test_saved = test_data.node_ids.detach().cpu().numpy().astype(int)

pred_path = os.path.join(GNN_SAVE_DIR, "gnn_test_preds.npz")
np.savez_compressed(
    pred_path,
    txid=txid_test_saved,
    proba=proba_test,
    threshold=float(best_threshold_pre),
)
print("Saved TEST preds:", pred_path, "shape:", proba_test.shape)


Saved checkpoint: gnn_saved\txs_gnn_checkpoint.pt
Saved scalers (*.pkl) into gnn_saved
Saved TEST preds: gnn_saved\gnn_test_preds.npz shape: (16346,)
